# Word Embedding for Sequence Processing

**The goal of this practical is to use pre-trained word embedding for adressing the sequence prediction tasks studied in week 2: PoS and chunking.**

In [1]:
import numpy as np
import gensim.downloader as api
from gensim.models import KeyedVectors

## 0) Loading PoS (or chunking) datasets (small or large)

In [2]:
def load(filename):
    listeDoc = list()
    with open(filename, "r") as f:
        doc = list()
        for ligne in f:
            #print "l : ",len(ligne)," ",ligne
            if len(ligne) < 2: # fin de doc
                listeDoc.append(doc)
                doc = list()
                continue
            mots = ligne.replace("\n","").split(" ")
            doc.append((mots[0],mots[1])) # mettre mots[2] à la place de mots[1] pour le chuncking
    return listeDoc

In [3]:
bSmall = True

if(bSmall==True):
    filename = "datasets/conll2000/chtrain.txt"
    filenameT = "datasets/conll2000/chtest.txt"

else:
    # Larger corpus .
    filename = "datasets/conll2000/train.txt"
    filenameT = "datasets/conll2000/test.txt"

alldocs = load(filename)
alldocsT = load(filenameT)

print(len(alldocs)," docs read")
print(len(alldocsT)," docs (T) read")

823  docs read
77  docs (T) read


# 1) Word embedding for classifying each word

### Pre-trained word2vec

In [4]:
import gensim.downloader as api
bload = True
fname = "word2vec-google-news-300"
sdir = "" # Change

if(bload==True):
    wv_pre_trained = KeyedVectors.load(sdir+fname+".dat")
else:
    wv_pre_trained = api.load(fname)
    wv_pre_trained.save(sdir+fname+".dat")

### Some token on the dataset are missing, we will encode them with a random vector
This is sub-optimal, but we need to do something

In [5]:
def randomvec():
    default = np.random.randn(300)
    default = default  / np.linalg.norm(default)
    return default

In [6]:
np.random.seed(seed=10) # seed the randomness

dictadd = dict()
cpt=0
for d in alldocs:
    cpt+=1
    #print(" ****** Document ******",cpt)
    for (x,pos) in d:
        if (not (x in wv_pre_trained) and not (x in dictadd)):
            #print(x," not in WE, adding it with random vector")
            dictadd[x] = randomvec()

for d in alldocsT:
    cpt+=1
    #print(" ****** TEST Document ******",cpt)
    for (x,pos) in d:
        if (not (x in wv_pre_trained) and not (x in dictadd)):
            #print(x," not in WE, adding it with random vector")
            dictadd[x] = randomvec()
            #wv_pre_trained.add_vector(x,randomvec())


### Add the (key-value) 'random' word embeddings for missing inputs

In [7]:
## YOUR CODE HERE
#print(dictadd)
keys = list(dictadd.keys())
values = list(dictadd.values())

wv_pre_trained.add_vectors(keys, values)

### Store the train and test datasets: a word embedding for each token in the sequences

In [8]:
wvectors = [wv_pre_trained.get_vector(x) for d in alldocs for x, pos in d] ## YOUR CODE HERE
wvectorsT = [wv_pre_trained.get_vector(x) for d in alldocsT for x, pos in d] ## YOUR CODE HERE

### Check the size of your train/test datasets

In [9]:
## YOUR CODE HERE
print(len(wvectors))
print(len(wvectorsT))

19172
1896


### Collecting train/test labels

In [10]:
# Labels train/test

buf2 = [[pos for m,pos in d ] for d in alldocs]
cles = []
[cles.extend(b) for b in buf2]
cles = np.unique(np.array(cles))
cles2ind = dict(zip(cles,range(len(cles))))
nCles = len(cles)
print(nCles," keys in the dictionary")

labels  = np.array([cles2ind[pos] for d in alldocs for (m,pos) in d ])
#np.array([cles2ind[pos] for (m,pos) in d for d in alldocs])
labelsT  = np.array([cles2ind.setdefault(pos,len(cles)) for d in alldocsT for (m,pos) in d ])

print(len(cles2ind)," keys in the dictionary")

42  keys in the dictionary
43  keys in the dictionary


In [11]:
print(labels.shape)
print(labelsT.shape)

(19172,)
(1896,)


### Train a Logistic Regression Model!
**And compare performances to the baseline and sequence models (HMM/CRF) or practical 2a**

In [12]:
## YOUR CODE HERE
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

#Logistic Regression
t = 1e-8
C=100.0
lr_clf = LogisticRegression(random_state=0, solver='lbfgs',max_iter=1000, tol=t, C=C)
lr_clf.fit(wvectors, labels)

pred_lrt = lr_clf.predict(wvectors)
pred_lr = lr_clf.predict(wvectorsT)

labs = list(set(labels))
labsT = list(set(labelsT))

print(f"Logistic Regression accuracy train={accuracy_score(labels, pred_lrt)}, accuracy test={accuracy_score(labelsT, pred_lr)}")
print(f"Logistic Regression f1_score train={f1_score(y_true=labels, y_pred=pred_lrt, labels=labs, average='micro')}, accuracy test={f1_score(y_true=labelsT, y_pred=pred_lr, labels=labsT, average='micro')}")
print(f"Logistic Regression precision train={precision_score(y_true=labels, y_pred=pred_lrt, labels=labs, average='micro')}, accuracy test={precision_score(y_true=labelsT, y_pred=pred_lr, labels=labsT, average='micro')}")
print(f"Logistic Regression recall train={recall_score(y_true=labels, y_pred=pred_lrt, labels=labs, average='micro')}, accuracy test={recall_score(y_true=labelsT, y_pred=pred_lr, labels=labsT, average='micro')}")

Logistic Regression accuracy train=0.9647924055914876, accuracy test=0.9003164556962026
Logistic Regression f1_score train=0.9647924055914876, accuracy test=0.9003164556962026
Logistic Regression precision train=0.9647924055914876, accuracy test=0.9003164556962026
Logistic Regression recall train=0.9647924055914876, accuracy test=0.9003164556962026


# 2) Using word embedding with CRF

## We will define the following features functions for CRF

In [29]:
def features_wv(sentence, index):
    v = wv_pre_trained.get_vector(sentence[index])
    d = {'f'+str(i):v[i] for i in range(300)}
    return d

def features_structural(sentence, index):
    return {
        'word': sentence[index],
        'is_first': index == 0,
        'is_last': index == len(sentence) - 1,
        'is_capitalized': sentence[index][0].upper() == sentence[index][0],
        'is_all_caps': sentence[index].upper() == sentence[index],
        'is_all_lower': sentence[index].lower() == sentence[index],
        'prefix-1': sentence[index][0],
        'prefix-2': sentence[index][:2],
        'prefix-3': sentence[index][:3],
        'suffix-1': sentence[index][-1],
        'suffix-2': sentence[index][-2:],
        'suffix-3': sentence[index][-3:],
        'prev_word': '' if index == 0 else sentence[index - 1],
        'next_word': '' if index == len(sentence) - 1 else sentence[index + 1],
        'has_hyphen': '-' in sentence[index],
        'is_numeric': sentence[index].isdigit(),
        ## We will define the following features functions for CRF
        ## We will define the following features functions for CRF   
        #'capitals_inside': sentence[index][1:].lower() != sentence[index][1:]
    }
    
def features_wv_plus_structural(sentence, index):
    v = wv_pre_trained.get_vector(sentence[index])
    d = {'f'+str(i):v[i] for i in range(300)}

    return {**d, **features_structural(sentence, index)}

## [Question]: explain what the 3 feature functions encode and what their differences are

## Feature function 1

First function creates a dictionnary with key-values corresponding to the index in the Word2Vec vector and and its value, that way the vector can be translated into something of the shape of a CRF.

## Feature function 2

The second one rather than using the vectorized word it incorporates an understading of the word in a "semantic" way, I would say that it's useful for words that share some similarities in suffix/prefix, that way the CRF can understand those similarities.

## Feature function 3

Last function is a combination of both.


### You can now train a CRF with the 3 features and analyse the results

In [26]:
from nltk.tag.crf import CRFTagger

tagger = CRFTagger(feature_func=features_structural)
tagger.train(alldocs, 'out/crf.model')
test = [[m for m,pos in d ] for d in alldocsT]
preds = tagger.tag_sents(test)
preds = np.array([pred for d in preds for m, pred in d])
reals = np.array([real for d in alldocsT for m, real in d])
print(preds[:10])
print(reals[:10])
correct = (preds == reals).sum()
correct/len(reals) 

['NN' 'IN' 'DT' 'NN' 'VBZ' 'RB' 'VBN' 'TO' 'VB' 'DT']
['NN' 'IN' 'DT' 'NN' 'VBZ' 'RB' 'VBN' 'TO' 'VB' 'DT']


np.float64(0.9140295358649789)

In [27]:
tagger = CRFTagger(feature_func=features_wv)
tagger.train(alldocs, 'out/crf.model')
test = [[m for m,pos in d ] for d in alldocsT]
preds = tagger.tag_sents(test)
preds = np.array([pred for d in preds for m, pred in d])
reals = np.array([real for d in alldocsT for m, real in d])
print(preds[:10])
print(reals[:10])
correct = (preds == reals).sum()
correct/len(reals) 

['NN' 'IN' 'DT' 'NN' 'VBZ' 'RB' 'VBN' 'TO' 'VB' 'DT']
['NN' 'IN' 'DT' 'NN' 'VBZ' 'RB' 'VBN' 'TO' 'VB' 'DT']


np.float64(0.9140295358649789)

In [30]:
tagger = CRFTagger(feature_func=features_wv_plus_structural)
tagger.train(alldocs, 'out/crf.model')
test = [[m for m,pos in d ] for d in alldocsT]
preds = tagger.tag_sents(test)
preds = np.array([pred for d in preds for m, pred in d])
reals = np.array([real for d in alldocsT for m, real in d])
print(preds[:10])
print(reals[:10])
correct = (preds == reals).sum()
correct/len(reals) 

['NN' 'IN' 'DT' 'NN' 'VBZ' 'RB' 'VBN' 'TO' 'VB' 'DT']
['NN' 'IN' 'DT' 'NN' 'VBZ' 'RB' 'VBN' 'TO' 'VB' 'DT']


np.float64(0.9583333333333334)